# Molecular Devices FilterMax F5 connection test

Connect to the reader through PyLabRobot on `/dev/ttyS0` using **38400 baud, 8 data bits, no parity, 1 stop bit, and no software or hardware flow control (8N1)**. PySerial applies these settings when it opens the port; running `stty` first is not required.

In [1]:
%load_ext autoreload
%autoreload 2

import asyncio

from pylabrobot.io.serial import Serial
from pylabrobot.plate_reading.molecular_devices import MolecularDevicesBackend

In [2]:
class FilterMaxF5ProbeBackend(MolecularDevicesBackend):
    """Molecular Devices backend configured for the FilterMax F5 serial link."""

    def __init__(self, port: str = "/dev/ttyS0") -> None:
        super().__init__(port=port)
        # MolecularDevicesBackend defaults to 9600 baud. Replace its transport
        # with the settings confirmed using stty: 38400, 8N1, no flow control.
        self.io = Serial(
            human_readable_device_name="Molecular Devices FilterMax F5",
            port=port,
            baudrate=38_400,
            bytesize=8,
            parity="N",
            stopbits=1,
            timeout=0.2,
            write_timeout=1,
            rtscts=False,
            dsrdtr=False,
        )


PORT = "/dev/ttyS0"
backend = FilterMaxF5ProbeBackend(port=PORT)

In [3]:
# Open only the PyLabRobot serial transport. Do not call backend.setup() yet:
# MolecularDevicesBackend.setup() assumes the SpectraMax-style `!` protocol.
async def raw_exchange(payload: bytes, response_window: float = 2.0) -> bytes:
    await backend.io.reset_input_buffer()
    await backend.io.write(payload)

    response = bytearray()
    deadline = asyncio.get_running_loop().time() + response_window
    while asyncio.get_running_loop().time() < deadline:
        response.extend(await backend.io.read(256))
    return bytes(response)


try:
    await backend.io.setup()
    print(f"Serial port opened on {backend.io.port}")
    print("Requested settings:", backend.io.serialize())
    raw_response = await raw_exchange(b"!\r")
    print("Raw response to b'!\\r':", repr(raw_response))
finally:
    await backend.stop()

2026-07-02 11:35:21,200 - pylabrobot.io.serial - INFO - Using explicitly provided port: /dev/ttyS0 (for VID=None, PID=None)


Serial port opened on /dev/ttyS0
Requested settings: {'human_readable_device_name': 'Molecular Devices FilterMax F5', 'port': '/dev/ttyS0', 'baudrate': 38400, 'bytesize': 8, 'parity': 'N', 'stopbits': 1, 'write_timeout': 1, 'timeout': 0.2, 'rtscts': False, 'dsrdtr': False}
Raw response to b'!\r': b''


Interpretation:

- `Serial port opened` confirms only the Linux/PySerial connection.
- A raw response ending in `>` indicates that the inherited Molecular Devices command protocol may be usable.
- `b''` means the computer received no bytes. Verify the original FilterMax RS-232 cable, instrument power/initialization, and the physical port before changing PyLabRobot commands.
- Non-empty data without the expected `>` framing indicates that a FilterMax-specific protocol implementation is required.

In [4]:
# Do not send motion or assay commands until the FilterMax F5 protocol has
# been identified and a read-only identity/status query succeeds.

In [ ]:
# # tested on 20260702 without success. A specific 9-pin serial cable did not accompany this purchase. 
# 2026-07-02 11:35:21,200 - pylabrobot.io.serial - INFO - Using explicitly provided port: /dev/ttyS0 (for VID=None, PID=None)
# Serial port opened on /dev/ttyS0
# Requested settings: {'human_readable_device_name': 'Molecular Devices FilterMax F5', 'port': '/dev/ttyS0', 'baudrate': 38400, 'bytesize': 8, 'parity': 'N', 'stopbits': 1, 'write_timeout': 1, 'timeout': 0.2, 'rtscts': False, 'dsrdtr': False}
# Raw response to b'!\r': b''

# Interpretation:

# - `Serial port opened` confirms only the Linux/PySerial connection.
# - A raw response ending in `>` indicates that the inherited Molecular Devices command protocol may be usable.
# - `b''` means the computer received no bytes. Verify the original FilterMax RS-232 cable, instrument power/initialization, and the physical port before changing PyLabRobot commands.
# - Non-empty data without the expected `>` framing indicates that a FilterMax-specific protocol implementation is required.